In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_13_try_5.pickle

In [ ]:
%%RecordEvent
%%time
### cell 14 ###

### cell 14 – optimized for cudf

# 1) group and count on GPU
# 2) filter on GPU (only the needed rows and columns)
# 3) build the message Series on GPU
# 4) attach it as a column, transfer once via Arrow, then print

df_grouped = (
    flights_df
      .groupby(['carrier', 'flight', 'dest'])
      .size()
      .reset_index(name='Size')
)
filtered = df_grouped[df_grouped['Size'] == 365][['carrier', 'flight', 'dest']]

msgs = (
    "Carrier: " + filtered['carrier']
    + ", Flight: " + filtered['flight'].astype(str)
    + ", Destination: " + filtered['dest']
)

# Assign the GPU‐built messages back into the DataFrame
filtered = filtered.assign(msg=msgs)

# One GPU→CPU transfer: convert the Series to Arrow, then to a Python list
msgs_list = filtered['msg'].to_arrow().to_pylist()

# Print all lines at once
print("\n".join(msgs_list))

In [ ]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_14_try_7.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:
with open(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/opt_cell_exec_info_14_try_7.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[14], f)

In [ ]:
opt_output = Out.get(4)